In [1]:
import numpy as np
import pandas as pd
import ipaddress
import os

def ue_ip_pool(subnet, n):
    net = ipaddress.ip_network(subnet)
    return [str(ip) for ip in list(net.hosts())[:n]]

def generate_slot_based_traffic_per_ue(
    T=10.0,                 # seconds
    slot_ms=1.0,
    gnb_ip="10.0.0.1",
    N_embb_ue=15,
    N_urllc_ue=5,
    embb_pkt_size=1500,
    urllc_pkt_size=32,
    embb_pkts_per_slot=1,
    urllc_prob_per_slot=0.06,
    out_dir="traffic_csv",
    seed=42
):
    rng = np.random.default_rng(seed)
    os.makedirs(out_dir, exist_ok=True)

    n_slots = int(T * 1000 / slot_ms)
    times = np.arange(n_slots) * slot_ms / 1000.0

    embb_ues = ue_ip_pool("10.0.1.0/24", N_embb_ue)
    urllc_ues = ue_ip_pool("10.0.2.0/24", N_urllc_ue)

    # -----------------------
    # eMBB UEs
    # -----------------------
    for i, ue_ip in enumerate(embb_ues, start=1):
        rows = []
        for t in times:
            length = embb_pkt_size if embb_pkts_per_slot > 0 else 0
            rows.append({
                "time": t,
                "slice": "eMBB",
                "direction": "DL",
                "src_ip": gnb_ip,
                "dst_ip": ue_ip,
                "length": length
            })
        df = pd.DataFrame(rows)
        df.to_csv(f"{out_dir}/embb_ue_{i}.csv", index=False)

    # -----------------------
    # URLLC UEs
    # -----------------------
    for i, ue_ip in enumerate(urllc_ues, start=1):
        rows = []
        for t in times:
            if rng.random() < urllc_prob_per_slot:
                length = urllc_pkt_size
            else:
                length = 0

            rows.append({
                "time": t,
                "slice": "URLLC",
                "direction": "DL",
                "src_ip": gnb_ip,
                "dst_ip": ue_ip,
                "length": length
            })
        df = pd.DataFrame(rows)
        df.to_csv(f"{out_dir}/urllc_ue_{i}.csv", index=False)


In [2]:
generate_slot_based_traffic_per_ue(
    T=10.0,
    N_embb_ue=1,
    N_urllc_ue=1,
    urllc_prob_per_slot=0.06,
    out_dir="traffic_1"
)